# Workflow Data Collection for Model Training

Collect and prepare training data from your workflow executions to improve model performance.

**100% Local - No API Keys Required**

## 1. Connect to Local Database

In [ ]:
!pip install psycopg2-binary pandas

In [ ]:
import psycopg2
import pandas as pd
import json

# Connect to local PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="workflow_manager",
    user="postgres",
    password="postgres"
)

print("Connected to database")

## 2. Extract Workflow Executions

In [ ]:
# Get successful workflow executions
query = """
SELECT 
    we.id,
    w.name as workflow_name,
    we.input_data,
    we.trace,
    we.output_data,
    we.status
FROM workflow_executions we
JOIN workflows w ON we.workflow_id = w.id
WHERE we.status = 'completed'
ORDER BY we.created_at DESC
LIMIT 100
"""

df = pd.read_sql_query(query, conn)
print(f"Found {len(df)} successful executions")
df.head()

## 3. Extract Planning Examples

In [ ]:
def extract_planning_examples(df):
    """Extract workflow planning examples"""
    examples = []
    
    for _, row in df.iterrows():
        if row['trace']:
            trace = json.loads(row['trace']) if isinstance(row['trace'], str) else row['trace']
            
            # Extract steps from trace
            steps = []
            for item in trace:
                if 'action' in item:
                    steps.append({
                        "step": item.get('step', 0),
                        "action": item.get('action'),
                        "tool": item.get('action')
                    })
            
            if steps:
                examples.append({
                    "instruction": f"Plan a {row['workflow_name']} workflow",
                    "input": json.dumps(row['input_data']),
                    "output": json.dumps(steps)
                })
    
    return examples

planning_examples = extract_planning_examples(df)
print(f"Extracted {len(planning_examples)} planning examples")
print("\nExample:")
print(json.dumps(planning_examples[0], indent=2) if planning_examples else "No examples found")

## 4. Extract Validation Examples

In [ ]:
# Get executions with errors (for validation training)
error_query = """
SELECT 
    we.id,
    we.trace,
    we.error_message
FROM workflow_executions we
WHERE we.status = 'failed'
ORDER BY we.created_at DESC
LIMIT 50
"""

error_df = pd.read_sql_query(error_query, conn)

def extract_validation_examples(df):
    """Extract validation examples from failed executions"""
    examples = []
    
    for _, row in df.iterrows():
        if row['trace'] and row['error_message']:
            trace = json.loads(row['trace']) if isinstance(row['trace'], str) else row['trace']
            
            for item in trace:
                if 'result' in item and 'error' in item['result'].lower():
                    examples.append({
                        "instruction": "Validate workflow execution result",
                        "input": f"Step: {item.get('action')}, Result: {item.get('result')}",
                        "output": json.dumps({
                            "valid": False,
                            "errors": [row['error_message']],
                            "suggestions": ["Review parameters and retry"]
                        })
                    })
    
    return examples

validation_examples = extract_validation_examples(error_df)
print(f"Extracted {len(validation_examples)} validation examples")

## 5. Extract Correction Examples

In [ ]:
def extract_correction_examples(df):
    """Extract self-correction examples"""
    examples = []
    
    for _, row in df.iterrows():
        if row['trace']:
            trace = json.loads(row['trace']) if isinstance(row['trace'], str) else row['trace']
            
            # Look for retry patterns
            for i, item in enumerate(trace):
                if item.get('attempt', 1) > 1:
                    # This was a retry - extract the correction
                    examples.append({
                        "instruction": "Correct workflow step parameters",
                        "input": f"Error: {item.get('error', 'Unknown error')}",
                        "output": json.dumps({
                            "corrected_params": item.get('parameters', {}),
                            "reasoning": "Applied suggested corrections"
                        })
                    })
    
    return examples

correction_examples = extract_correction_examples(df)
print(f"Extracted {len(correction_examples)} correction examples")

## 6. Combine and Save Training Data

In [ ]:
# Combine all examples
all_examples = planning_examples + validation_examples + correction_examples

print(f"\nTotal training examples: {len(all_examples)}")
print(f"- Planning: {len(planning_examples)}")
print(f"- Validation: {len(validation_examples)}")
print(f"- Correction: {len(correction_examples)}")

# Save to file
with open('workflow_training_data.json', 'w') as f:
    json.dump(all_examples, f, indent=2)

print("\nSaved to workflow_training_data.json")

## 7. Data Quality Analysis

In [ ]:
# Analyze data quality
def analyze_training_data(examples):
    """Analyze training data quality"""
    stats = {
        "total": len(examples),
        "avg_input_length": sum(len(e['input']) for e in examples) / len(examples) if examples else 0,
        "avg_output_length": sum(len(e['output']) for e in examples) / len(examples) if examples else 0,
        "unique_instructions": len(set(e['instruction'] for e in examples))
    }
    return stats

stats = analyze_training_data(all_examples)
print("\nData Quality Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:.2f}" if isinstance(value, float) else f"  {key}: {value}")

## 8. Create Synthetic Examples

In [ ]:
# Generate synthetic examples for common patterns
synthetic_examples = [
    {
        "instruction": "Plan a client onboarding workflow",
        "input": json.dumps({"client_name": "Example Corp", "email": "contact@example.com"}),
        "output": json.dumps([
            {"step": 1, "action": "Create Jira project", "tool": "jira_create_project"},
            {"step": 2, "action": "Create Stripe customer", "tool": "stripe_create_customer"},
            {"step": 3, "action": "Send welcome email", "tool": "send_email"}
        ])
    },
    {
        "instruction": "Plan an invoice processing workflow",
        "input": json.dumps({"invoice_url": "https://example.com/invoice.pdf"}),
        "output": json.dumps([
            {"step": 1, "action": "Extract invoice data", "tool": "ocr_extract"},
            {"step": 2, "action": "Validate data", "tool": "validate_invoice"},
            {"step": 3, "action": "Create payment", "tool": "stripe_create_payment"}
        ])
    },
    {
        "instruction": "Validate workflow execution result",
        "input": "Step: Create Jira project, Result: Success - Project TEST created",
        "output": json.dumps({"valid": True, "errors": [], "suggestions": []})
    }
]

# Add synthetic examples
all_examples.extend(synthetic_examples)

# Save updated dataset
with open('workflow_training_data_complete.json', 'w') as f:
    json.dump(all_examples, f, indent=2)

print(f"Added {len(synthetic_examples)} synthetic examples")
print(f"Total examples: {len(all_examples)}")

## 9. Export for Fine-tuning

In [ ]:
# Format for different training frameworks

# Format 1: Instruction format
def format_for_instruction_tuning(examples):
    formatted = []
    for ex in examples:
        formatted.append({
            "text": f"""### Instruction:
{ex['instruction']}

### Input:
{ex['input']}

### Output:
{ex['output']}"""
        })
    return formatted

# Format 2: Chat format
def format_for_chat_tuning(examples):
    formatted = []
    for ex in examples:
        formatted.append({
            "messages": [
                {"role": "system", "content": "You are a workflow automation agent."},
                {"role": "user", "content": f"{ex['instruction']}\n\n{ex['input']}"},
                {"role": "assistant", "content": ex['output']}
            ]
        })
    return formatted

# Save both formats
instruction_format = format_for_instruction_tuning(all_examples)
with open('training_data_instruction.json', 'w') as f:
    json.dump(instruction_format, f, indent=2)

chat_format = format_for_chat_tuning(all_examples)
with open('training_data_chat.json', 'w') as f:
    json.dump(chat_format, f, indent=2)

print("Exported training data in multiple formats:")
print("- training_data_instruction.json (for instruction tuning)")
print("- training_data_chat.json (for chat tuning)")

## Summary

### What We Did
1. ✅ Connected to local database
2. ✅ Extracted workflow executions
3. ✅ Created planning examples
4. ✅ Created validation examples
5. ✅ Created correction examples
6. ✅ Added synthetic examples
7. ✅ Exported in multiple formats

### Next Steps
1. Use the exported data in notebook 02 for fine-tuning
2. Collect more real-world execution data
3. Iterate on model performance
4. Deploy fine-tuned model

**All data stays local - complete privacy and control**